In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv(r"../data/fraudTrain.csv")


df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'], format='%Y-%m-%d %H:%M:%S')

In [ ]:
counts = df['is_fraud'].value_counts()
percentages = df['is_fraud'].value_counts(normalize=True) * 100

imbalance_summary = pd.DataFrame({'Count': counts, 'Percentage': percentages})
print("Class Imbalance Summary:")
print(imbalance_summary)

# 1. Print the exact numbers to see the gap
print(df["is_fraud"].value_counts())

# 2. Print the percentages
print(df["is_fraud"].value_counts(normalize=True) * 100)

# 3. Plot the massive imbalance
sns.countplot(x="is_fraud", data=df)
plt.title("Credit Card Fraud Distribution (0 = Safe, 1 = Fraud)")
plt.yscale("log")  # Optional: use log scale so the tiny fraud bar is actually visible
plt.show()


In [ ]:
df[df["is_fraud"] == 1]["amt"].hist(bins=50)
fraud_amt_dist = df[df["is_fraud"] == 1]["amt"].describe()
median_fraud = df[df["is_fraud"] == 1]["amt"].median()
print(f"Median fraudulent transaction amount: {median_fraud}")
print(fraud_amt_dist)

plt.title("Distribution of Fraudulent Transaction Amounts")
plt.xlabel("Amount")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df[df["is_fraud"] == 1]["trans_date_trans_time"].hist(bins=25)
plt.title("Distribution of Fraudulent Transaction Times")
plt.xlabel("Transaction Time")
plt.ylabel("Frequency")
plt.xticks(rotation=45)
plt.show()

In [ ]:
from geopy.distance import geodesic

m_lat, m_long, c_lat, c_long = map(
    np.radians,
    [df["merch_lat"], df["merch_long"], df["lat"], df["long"]],
)
dlat = c_lat - m_lat
dlon = c_long - m_long

a = np.sin(dlat / 2) ** 2 + np.cos(m_lat) * np.cos(c_lat) * np.sin(dlon / 2) ** 2
c = 2 * np.arcsin(np.sqrt(a))

# 3. Assign the distance and round it
df["distance"] = (3958.8 * c).round(2)
fraud_dist_corr = df[["distance", "is_fraud"]].corr().iloc[0, 1]
print(f"Correlation between distance and fraud: {fraud_dist_corr}")
# 4. Display the results
display(df)



In [77]:
# Generate a unique ID for each unique person + zip combo
df["customer_id"] = df.groupby(["first", "last", "zip", "dob", "cc_num"]).ngroup()
fraud_count = df.groupby('customer_id')['is_fraud'].sum().reset_index()
fraud_count.columns = ["customer_id", "fraud_occurrences"]
# Group by customer profile and calculate multiple metrics at once
customer_df = (
    df.groupby(["customer_id", "first", "last", "zip", "dob", "gender", "job", "lat", "long"])
    .agg(
        unique_cc_count=("cc_num", "nunique"),
        total_fraud_events=("is_fraud", "sum"),
        total_spend=("amt", "sum"),  # Change 'amt' to your spending column name
        last_active_date=(
            "trans_date_trans_time",
            "max",
        ),  
        total_transactions=("trans_num", "count")
   
    )
    .reset_index()
)


display(customer_df)

,customer_id,first,last,zip,dob,gender,job,lat,long,unique_cc_count,total_fraud_events,total_spend,last_active_date,total_transactions
0,0,Aaron,Murray,64659,1974-12-23,M,Tourist information centre manager,39.7795,-93.3014,1,8,204477.60,2020-06-21 09:26:55,2050
1,1,Aaron,Pena,22015,1950-11-27,M,Health visitor,38.7894,-77.2818,1,0,97973.69,2020-06-21 11:36:41,1476
2,2,Aaron,Rogers,69201,1945-03-15,M,Network engineer,42.8062,-100.6215,1,12,38418.91,2020-06-21 02:22:34,508
3,3,Aaron,Stewart,4364,1995-04-22,M,Advertising account planner,44.3229,-69.9576,1,8,28704.06,2020-06-20 20:19:04,537
4,4,Adam,Keller,36775,1932-09-17,M,Learning disability nurse,32.2844,-86.9920,1,14,33310.58,2020-06-20 13:52:33,521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
978,978,William,Thompson,12575,1937-03-17,M,Building surveyor,41.4575,-74.1659,1,12,133746.90,2020-06-21 10:15:35,2052
979,979,Willie,Jordan,71277,1957-08-08,M,"Psychologist, forensic",32.9550,-92.5870,1,4,63010.83,2020-06-21 08:25:28,1038
980,980,Xavier,Beltran,40914,1984-06-04,M,"Psychologist, forensic",37.1046,-83.5706,1,13,151215.59,2020-06-21 11:40:31,1516
981,981,Zachary,Allen,52576,1969-07-24,M,Commercial horticulturist,41.2001,-92.1354,1,8,95744.47,2020-06-21 02:40:39,1523
